## Paths and executables

In [1]:
conda activate ppanggolin
ppanggolin -v
#conda deactivate

ppanggolin 2.2.5


In [2]:
# Input folders
genomesFolder="data/00_fasta/subset"

# Output folders
outputFolder="data/01_pangenome"
mkdir -p $outputFolder

# Root folder
root=$(pwd)

# workfolder
workfolder="/mnt/c/Users/robbe/Documents/KULeuven/Bioinformatics_Thesis/PaRMMoSaHN"
cd $workfolder



## Parameters

In [3]:
cluster_id=0.80
cluster_cov=0.60
n_cores=1
label="clostridiales"

## Construct pangenome

In [ ]:
conda activate ppanggolin

Assuming you already have annotated your genomes using Bakta or obtained annotations from NCBI, put the GenBank proteome files (`*.gbff`) in some folder.

### Preventive cleanup of the annotations

In [4]:
# Remove problematic characters from GenBank files (’) 
find $genomesFolder/*.gbff | xargs -I % bash -c 'cat % | tr -d "’" > tmp && mv tmp %'

### Build!

In [5]:
paste <(dir -1 $genomesFolder | grep .gbff) <(find "$genomesFolder" | grep .gbff) > annotations.list

In [6]:
pwd

/mnt/c/Users/robbe/Documents/KULeuven/Bioinformatics_Thesis/PaRMMoSaHN


In [ ]:

ppanggolin annotate -f --anno annotations.list -o $outputFolder -c $n_cores

2025-11-12 11:57:53 utils.py:l239 INFO	Command: /home/robbe/miniconda3/envs/ppanggolin/bin/ppanggolin annotate -f --anno annotations.list -o data/01_pangenome -c 1
2025-11-12 11:57:53 utils.py:l242 INFO	PPanGGOLiN version: 2.2.5
2025-11-12 11:57:53 utils.py:l977 INFO	2 parameters have a non-default value.
2025-11-12 11:57:53 annotate.py:l1533 INFO	Reading annotations.list the list of genome files ...
 10%|████▏                                     | 1/10 [00:00<00:08,  1.12file/s]2025-11-12 11:57:55 annotate.py:l756 INFO	CDS 'GCF_000143685.1_13820' contains a 'transl_except' annotation ((pos:complement(3022521..3022523),aa:Sec)) in contig 'contig_1' in file 'data/00_fasta/subset/GCF_000143685.1.gbff'. PPanGGOLiN does not handle 'transl_except' annotations. This gene's protein sequence will likely contain an internal stop codon when translated with PPanGGOLiN.
2025-11-12 11:57:55 annotate.py:l756 INFO	CDS 'GCF_000143685.1_14090' contains a 'transl_except' annotation ((pos:complement(3076

In [ ]:
#from chatgpt for reference
# A step-by-step pangenome construction
# 1. Initialize the pangenome
ppanggolin init -i annotations.list -o $outputFolder -l $label -c $n_cores
# 2. Compute the pangenome
ppanggolin compute -o $outputFolder -c $n_cores
# 3. Cluster the pangenome
ppanggolin cluster -o $outputFolder -c $n_cores --id $cluster
# 4. Classify the gene families
ppanggolin classify -o $outputFolder -c $n_cores --cov $cluster_cov
# 5. Annotate the pangenome
ppanggolin annotate -f --anno annotations.list -o $outputFolder -c $n_cores

In [14]:
pwd

: 1

In [ ]:
ppanggolin cluster -f -p $workFolder/pangenome.h5 --identity $cluster_id --coverage $cluster_cov -c $n_cores \
--tmpdir /mnt/DATA/tmp

2025-10-06 13:18:45 utils.py:l168 INFO	Command: /home/lucas/bin/anaconda3/envs/ppanggolin/bin/ppanggolin cluster -f -p 01-pangenome/pangenome.h5 --identity 0.80 --coverage 0.60 -c 14 --tmpdir /mnt/DATA/tmp
2025-10-06 13:18:45 utils.py:l169 INFO	PPanGGOLiN version: 2.0.5
2025-10-06 13:18:45 utils.py:l722 INFO	4 parameters have a non-default value.
2025-10-06 13:18:45 readBinaries.py:l94 INFO	Getting the current pangenome status
2025-10-06 13:18:45 readBinaries.py:l236 INFO	Extracting and writing CDS sequences from a /mnt/DATA/PhD/WPs/WP1/clostridia_models_9090/01-pangenome/pangenome.h5 file to a fasta file...
100%|█████████████████████████████| 781847/781847 [00:04<00:00, 176643.62gene/s]
2025-10-06 13:19:09 cluster.py:l317 INFO	Clustering all of the genes sequences...
2025-10-06 13:19:09 cluster.py:l90 INFO	Creating sequence database...
2025-10-06 13:19:19 cluster.py:l97 INFO	Clustering sequences...
2025-10-06 13:29:39 cluster.py:l103 INFO	Extracting cluster representatives...
2025-10-

In [ ]:
no_genomes=$(ppanggolin info -p $workFolder/pangenome.h5 --content | grep Genomes | grep -Eo '[0-9]+')

In [ ]:
ppanggolin fasta -f -p $workFolder/pangenome.h5 -o $workFolder --prot_families softcore --soft_core $(echo "scale=5; 2/$no_genomes" | bc)

2025-10-06 13:54:51 utils.py:l168 INFO	Command: /home/lucas/bin/anaconda3/envs/ppanggolin/bin/ppanggolin fasta -f -p 01-pangenome/pangenome.h5 -o 01-pangenome --prot_families softcore --soft_core .00956
2025-10-06 13:54:51 utils.py:l169 INFO	PPanGGOLiN version: 2.0.5
2025-10-06 13:54:51 utils.py:l722 INFO	4 parameters have a non-default value.
2025-10-06 13:54:51 readBinaries.py:l94 INFO	Getting the current pangenome status
2025-10-06 13:54:51 readBinaries.py:l715 INFO	Reading pangenome annotations...
100%|███████████████████████████████| 19693/19693 [00:00<00:00, 226947.44gene/s]
2025-10-06 13:55:01 readBinaries.py:l730 INFO	Reading pangenome gene families...
100%|███████████████████████| 413540/413540 [00:05<00:00, 80937.43gene family/s]
2025-10-06 13:55:14 writeSequences.py:l109 INFO	Writing the representative amino acid sequences of the gene families in softcore genome, that are present in more than 0.00956 of genomes
100%|████████████████████| 114661/114661 [00:00<00:00, 295490.80

In [ ]:
mv $workFolder/softcore_protein_families.faa $workFolder/$label.faa

In [ ]:
conda deactivate